# 🏦 Multi-Agent Workflow: Insurance Claims Processing

This notebook demonstrates how to build a **multi-agent workflow** using Azure AI Agent Service's `WorkflowAgentDefinition` with **YAML-based workflow orchestration** for automated insurance claims processing.

## What You'll Learn

- How to create specialist agents with specific roles
- How to define a **workflow YAML** that orchestrates multiple agents
- How to use `WorkflowAgentDefinition` for declarative multi-agent coordination
- How to process streaming workflow events

## Architecture Overview

The system uses a **declarative YAML workflow** that coordinates:
1. **Validity Agent**: Assesses claim validity
2. **Department Agent**: Assigns the appropriate department
3. **Payout Agent**: Estimates payout range
4. **Orchestrator Agent**: Synthesizes all assessments

### Workflow Pattern (YAML-based)

```yaml
kind: workflow
trigger: OnConversationStart
actions:
  - InvokeAzureAgent (validity)
  - InvokeAzureAgent (department)  
  - InvokeAzureAgent (payout)
  - InvokeAzureAgent (orchestrator)
```

This approach demonstrates a multi-agent workflow pattern for complex business processes.

## 📦 Import Required Libraries and Setup Environment

This cell imports all the necessary libraries for building our multi-agent insurance claims processing system and loads environment variables from the `.env` file. We need:

- **Azure AI Agents SDK**: To create and manage multiple AI agents
- **Azure Identity**: For authentication with Azure services
- **Environment variables**: Project endpoint and model deployment details

In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WorkflowAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import find_dotenv, load_dotenv


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")
load_dotenv(dotenv_path)

project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
required_settings = {
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_deployment,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

print("Required environment variables loaded")

## 🎯 Define Specialist Agent Instructions

Now we'll define the instructions for each of our four specialist agents. Each agent has a specific role in the insurance claims processing workflow.

### Claim Validity Agent

This agent analyzes claims to determine their validity based on policy coverage, documentation, and claim details.

In [ ]:
# Claim Validity Agent definition
validity_agent_name = "claim-validity-agent"
validity_agent_instructions = """
Assess whether an insurance claim is valid based on its description and coverage details.

Respond with one of the following statuses:
- Valid: Claim is covered under policy and documentation is complete
- Requires Review: Additional documentation or investigation needed
- Denied: Claim is not covered or policy exclusions apply

Only output the validity status and a very brief explanation of your determination.
"""

### Department Assignment Agent

This agent determines which claims department should handle each claim based on the type of insurance and nature of the claim.

In [ ]:
# Department Assignment Agent definition
department_agent_name = "department-assignment-agent"
department_agent_instructions = """
Decide which claims department should handle each insurance claim.

Choose from the following departments:
- Auto Claims: Vehicle accidents, theft, damage
- Home Claims: Property damage, theft, liability
- Life Claims: Death benefits, policy payouts
- Health Claims: Medical expenses, hospitalization
- Commercial Claims: Business-related insurance claims

Base your answer on the type of incident described. Respond with the department name and a very brief explanation.
"""

### Payout Estimation Agent

This agent estimates the potential payout amount based on the claim details, policy limits, and deductibles.

In [ ]:
# Payout Estimation Agent definition
payout_agent_name = "payout-estimation-agent"
payout_agent_instructions = """
Estimate the potential payout range for each insurance claim.

Use the following scale:
- Low: Under $5,000 - minor repairs, small medical expenses
- Medium: $5,000-$25,000 - significant repairs, moderate medical treatment
- High: Over $25,000 - major damage, extensive medical care, total loss

Base your estimate on the severity of the incident described. Respond with the payout level and a brief justification.
"""

### Claims Orchestrator Agent Instructions

The orchestrator agent coordinates all specialist agent outputs and provides a comprehensive claims processing recommendation.

In [ ]:
# Instructions for the orchestrator claims processing agent
orchestrator_agent_name = "claims-orchestrator-agent"
orchestrator_agent_instructions = """
You are a senior claims analyst who synthesizes assessments from multiple specialist agents.

When you receive an insurance claim, review all the context from the conversation which includes:
1. The original claim details
2. Validity assessment from the validity specialist
3. Department assignment from the department specialist
4. Payout estimation from the payout specialist

Your job is to:
1. Review the original claim and all specialist assessments in the conversation
2. Identify any inconsistencies or concerns
3. Provide a final comprehensive claims processing recommendation
4. Include next steps for the claims adjuster

Format your response as a structured claims report with clear sections:
- CLAIM SUMMARY
- VALIDITY STATUS
- ASSIGNED DEPARTMENT
- PAYOUT ESTIMATE
- FINAL RECOMMENDATION
- NEXT STEPS
"""

## 🔗 Connect to Azure AI Agent Service

This cell establishes a connection to the Azure AI Agent Service using our project endpoint and credentials. This client will be used to create and manage all our insurance claims processing agents.

In [ ]:
# Connect to the AIProjectClient using AzureCliCredential
credential = AzureCliCredential()
print("🔐 Using AzureCliCredential for authentication...")

# Initialize the project client with allow_preview for workflow agents
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
    allow_preview=True
)

# Get OpenAI client for conversations and responses
openai_client = project_client.get_openai_client()

print("✅ AIProjectClient initialized with AzureCliCredential (preview features enabled)")
print("✅ OpenAI client ready for conversations")

## 🤖 Create Multi-Agent Insurance Claims System

This cell creates all agents in our multi-agent system:

1. **Three Specialist Agents**: Validity, Department, and Payout assessment agents
2. **One Orchestrator Agent**: Synthesizes specialist assessments into a final recommendation

Each specialist agent is created with its specific instructions and model configuration.

In [ ]:
agent_specs = [
    (validity_agent_name, validity_agent_instructions),
    (department_agent_name, department_agent_instructions),
    (payout_agent_name, payout_agent_instructions),
    (orchestrator_agent_name, orchestrator_agent_instructions),
]
created_agents = []

try:
    for agent_name, instructions in agent_specs:
        agent = project_client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(
                model=model_deployment,
                instructions=instructions,
            ),
        )
        created_agents.append(agent)
        print(f"Created {agent.name}, version {agent.version}")
except Exception:
    for created_agent in reversed(created_agents):
        try:
            project_client.agents.delete_version(
                agent_name=created_agent.name,
                agent_version=created_agent.version,
            )
        except Exception as cleanup_error:
            print(f"Rollback failed for {created_agent.name}: {cleanup_error}")
    raise

validity_agent, department_agent, payout_agent, orchestrator_agent = created_agents

## 📝 Define Workflow YAML

Now we'll create a **declarative YAML workflow** that orchestrates our specialist agents. The workflow:

1. Receives a claim as input
2. Passes the claim through each specialist agent sequentially
3. Collects assessments in variables
4. Sends the combined assessments to the orchestrator for final synthesis

This is the key differentiator - instead of manual Python orchestration, we define the workflow declaratively!

In [ ]:
# Define the workflow YAML that orchestrates our specialist agents
# This YAML defines the sequential flow: Validity -> Department -> Payout -> Orchestrator
# All agents share the same conversation so orchestrator sees full history

workflow_yaml = f"""
kind: workflow
trigger:
  kind: OnConversationStart
  id: claims_processing_workflow
  actions:
    # Store the incoming claim in a variable
    - kind: SetVariable
      id: set_claim_input
      variable: Local.LatestMessage
      value: "=UserMessage(System.LastMessageText)"

    # Step 1: Invoke Validity Agent in MAIN conversation
    - kind: InvokeAzureAgent
      id: validity_assessment
      description: Assess claim validity
      agent:
        name: {validity_agent.name}
      input:
        messages: "=Local.LatestMessage"

    # Step 2: Invoke Department Agent in MAIN conversation
    - kind: InvokeAzureAgent
      id: department_assignment
      description: Assign department
      agent:
        name: {department_agent.name}
      input:
        messages: "=Local.LatestMessage"

    # Step 3: Invoke Payout Agent in MAIN conversation
    - kind: InvokeAzureAgent
      id: payout_estimation
      description: Estimate payout
      agent:
        name: {payout_agent.name}
      input:
        messages: "=Local.LatestMessage"

    # Step 4: Invoke Orchestrator in MAIN conversation
    # Orchestrator sees full conversation history with all specialist assessments
    - kind: InvokeAzureAgent
      id: orchestrator_synthesis
      description: Synthesize all assessments into final report
      agent:
        name: {orchestrator_agent.name}
      input:
        messages:
          - role: user
            content: "Now synthesize all the above assessments into a comprehensive claims report."

    # End the workflow
    - kind: EndConversation
      id: end_workflow
"""

print("📋 Workflow YAML defined successfully!")
print("\n🔄 Workflow Steps:")
print("   1️⃣ Receive claim input")
print("   2️⃣ Invoke Validity Agent → assess claim")
print("   3️⃣ Invoke Department Agent → assign department")
print("   4️⃣ Invoke Payout Agent → estimate payout")
print("   5️⃣ Invoke Orchestrator Agent → synthesize all (sees full conversation)")
print("   6️⃣ End workflow")
print("\n💡 All agents run in the MAIN conversation so orchestrator sees full history")

## 🏗️ Create Workflow Agent

Now we'll create a **WorkflowAgentDefinition** using our YAML workflow. This creates a single "workflow agent" that internally orchestrates all four specialist agents.

In [ ]:
try:
    workflow_agent = project_client.agents.create_version(
        agent_name="claims-processing-workflow",
        definition=WorkflowAgentDefinition(workflow=workflow_yaml),
    )
except Exception:
    rollback_errors = []
    for created_agent in reversed(created_agents):
        try:
            project_client.agents.delete_version(
                agent_name=created_agent.name,
                agent_version=created_agent.version,
            )
        except Exception as cleanup_error:
            rollback_errors.append(f"{created_agent.name}: {cleanup_error}")
    if rollback_errors:
        print("Rollback failures:\n" + "\n".join(rollback_errors))
    raise

print(f"Created workflow agent {workflow_agent.name}, version {workflow_agent.version}")

## 📋 Define Sample Insurance Claims

Let's define sample insurance claims to process through our workflow.

In [ ]:
# Sample insurance claims for demonstration
claims = [
    """Claim ID: CLM-2024-001
Policy Type: Auto Insurance
Incident: Vehicle collision at intersection on January 15, 2024.
Description: The insured's vehicle was struck by another car running a red light.
Police report filed. No injuries reported. Vehicle requires bumper replacement and 
alignment repair. Estimated repair cost: $3,200. Deductible: $500.""",
    
    """Claim ID: CLM-2024-002
Policy Type: Home Insurance  
Incident: Water damage from burst pipe on February 3, 2024.
Description: Frozen pipe burst in the basement causing flooding. Damage to flooring,
drywall, and personal belongings. Professional remediation required. 
Estimated damage: $15,000. Policy coverage limit: $250,000.""",
]

print(f"📋 Loaded {len(claims)} sample insurance claims")
for i, claim in enumerate(claims, 1):
    claim_id = claim.split("Claim ID:")[1].split("\n")[0].strip()
    policy_type = claim.split("Policy Type:")[1].split("\n")[0].strip()
    print(f"   • {claim_id} - {policy_type}")

## 🎯 Execute Workflow with Streaming Events

Now we'll run our workflow agent! This demonstrates:
- Creating a conversation for the workflow
- Streaming the workflow execution events
- Watching each agent action as it executes

In [ ]:
def process_claim_with_workflow(claim_text, claim_number):
    """Process one claim and return the workflow's final text output."""
    conversation = openai_client.conversations.create()
    print(f"Processing claim {claim_number} in conversation {conversation.id}")

    final_output = ""
    event_types = set()
    try:
        stream = openai_client.responses.create(
            conversation=conversation.id,
            extra_body={
                "agent_reference": {
                    "type": "agent_reference",
                    "name": workflow_agent.name,
                    "version": workflow_agent.version,
                }
            },
            input=claim_text,
            stream=True,
        )

        for event in stream:
            event_type = getattr(event, "type", None)
            if event_type:
                event_types.add(event_type)

            if event_type == "response.output_text.delta":
                delta = getattr(event, "delta", None)
                if delta:
                    print(delta, end="", flush=True)
                    final_output += delta
            elif event_type == "response.output_item.added":
                item = getattr(event, "item", None)
                if getattr(item, "type", None) == "workflow_action":
                    print(f"\nStarted workflow action: {getattr(item, 'action_id', 'unknown')}")
            elif event_type == "response.output_item.done":
                item = getattr(event, "item", None)
                if getattr(item, "type", None) == "workflow_action":
                    print(f"\nCompleted workflow action: {getattr(item, 'action_id', 'unknown')}")
            elif event_type == "response.completed" and not final_output:
                completed_response = getattr(event, "response", None)
                final_output = getattr(completed_response, "output_text", "") or ""

        if not final_output:
            raise RuntimeError(
                "Workflow completed without text output. "
                f"Observed event types: {sorted(event_types)}"
            )
        print("\n")
        return final_output
    finally:
        openai_client.conversations.delete(conversation.id)
        print(f"Deleted conversation {conversation.id}")


all_results = [
    process_claim_with_workflow(claim, claim_number)
    for claim_number, claim in enumerate(claims, start=1)
]
print(f"Processed {len(all_results)} claims successfully")
print("This demonstration does not replace review by licensed claims adjusters.")

## 🧹 Clean Up Resources

This cell deletes all the agents we created to avoid leaving resources running in Azure. It's important to clean up agents after use to prevent unnecessary costs.

In [ ]:
cleanup_errors = []

for agent in [
    workflow_agent,
    orchestrator_agent,
    payout_agent,
    department_agent,
    validity_agent,
]:
    try:
        project_client.agents.delete_version(
            agent_name=agent.name,
            agent_version=agent.version,
        )
        print(f"Deleted {agent.name} version {agent.version}")
    except Exception as error:
        cleanup_errors.append(f"{agent.name} version {agent.version}: {error}")

openai_client.close()
project_client.close()
credential.close()

if cleanup_errors:
    raise RuntimeError("Cleanup failures:\n" + "\n".join(cleanup_errors))